# Chapter 9 - Lab 3: <font color='blue'>Underwriting with Parallel Data Enrichment</font>

**<font color='purple'>Goal</font>**:
Build an **underwriting pipeline** that decides whether to accept a new policy applicant, and what premium to charge. The architecture is a **sequential + parallel hybrid** (Ch 7, §3.2):

* **Three enrichment agents** run in parallel via `asyncio.gather` — credit check, driving   record, property valuation.
* A **risk assessment** step then synthesises the enrichment data into a risk score, premium,   and accept / refer decision.

This lab is the smallest of the three — and the most directly useful as a starting point for a real onboarding workflow.

**<font color='purple'>Tech stack</font>**:

* **Python `asyncio`** — parallel enrichment via `asyncio.gather`.
* **No external LLM calls** in the core path — the enrichment functions are deterministic stubs   so the lab runs offline. The notebook ends with a short note on how to swap stubs for real APIs.

## 1. Install packages

This lab uses only the standard library. We install nothing beyond what is already in the notebook environment.

In [ ]:
# No additional packages needed for this lab.

## 2. Define the three enrichment functions

Each function simulates a slow external API with `asyncio.sleep`. In production these would be real calls — to Experian, the DMV, and a property valuation provider — but their *signatures* (input dict → output dict) stay the same, so you can swap them in without touching the orchestrator.

In [ ]:
import asyncio

async def check_credit(applicant_id: str) -> dict:
    """Stub: external credit-bureau API."""
    await asyncio.sleep(1.0)  # simulate latency
    return {'applicant_id': applicant_id, 'credit_score': 742, 'credit_rating': 'good'}


async def check_driving_record(license_number: str) -> dict:
    """Stub: external DMV API."""
    await asyncio.sleep(1.2)
    return {
        'license': license_number,
        'violations': 0,
        'accidents_3yr': 1,
        'risk_tier': 'standard',
    }


async def check_property_value(address: str) -> dict:
    """Stub: external property-valuation API."""
    await asyncio.sleep(0.8)
    return {
        'address': address,
        'estimated_value': 425_000,
        'risk_zone': 'moderate',
        'year_built': 2005,
    }

## 3. Run all three enrichment calls in parallel

The wall-clock time is roughly `max(1.0, 1.2, 0.8) = 1.2s` instead of `1.0 + 1.2 + 0.8 = 3.0s`. On real APIs the savings are larger — credit-bureau checks often take 3–5 seconds each.

In [ ]:
async def enrich_application(application: dict) -> dict:
    print('Running parallel data enrichment...')
    credit, driving, prop = await asyncio.gather(
        check_credit(application['applicant_id']),
        check_driving_record(application['license_number']),
        check_property_value(application['address']),
    )
    print(f"  Credit score:  {credit['credit_score']}")
    print(f"  Driving:       {driving['violations']} violations, {driving['accidents_3yr']} accidents")
    print(f"  Property:      ${prop['estimated_value']:,}")
    return {'credit': credit, 'driving': driving, 'property': prop}

## 4. Risk assessment — synthesising the enrichment data

The risk model is intentionally simple: a base score adjusted by credit band and accident count, with a premium computed from the property value. In a real underwriter this would be a calibrated model, not a hand-tuned scorecard — but the *shape* of the pipeline (parallel enrichment → synthesis) is exactly the same.

In [ ]:
def assess_risk(enrichment: dict) -> dict:
    credit_score = enrichment['credit']['credit_score']
    accidents    = enrichment['driving']['accidents_3yr']
    prop_value   = enrichment['property']['estimated_value']

    base_score = 100
    if credit_score >= 750:
        base_score -= 10
    elif credit_score < 650:
        base_score += 20
    base_score += accidents * 15

    base_premium = prop_value * 0.004              # 0.4% of property value
    risk_multiplier = 1.0 + (base_score - 80) / 100
    annual_premium = round(base_premium * risk_multiplier, 2)

    risk_tier = (
        'preferred' if base_score < 70
        else 'standard' if base_score < 100
        else 'high_risk'
    )
    decision = 'accept' if base_score < 120 else 'refer_to_senior'

    return {
        'risk_score': base_score,
        'risk_tier': risk_tier,
        'annual_premium': annual_premium,
        'decision': decision,
    }

## 5. Run the underwriting pipeline

In [ ]:
application = {
    'applicant_id': 'APP-2026-001',
    'name': 'James Rodriguez',
    'license_number': 'DL-NY-8834521',
    'address': '42 Oak Street, Westchester, NY 10601',
    'coverage_requested': 'homeowners + auto bundle',
}

print(f"Underwriting application: {application['name']}\n")

enrichment = await enrich_application(application)

print('\nAssessing risk...')
risk = assess_risk(enrichment)

print('\n' + '=' * 40)
print('  UNDERWRITING DECISION')
print('=' * 40)
print(f"  Risk score:     {risk['risk_score']}")
print(f"  Risk tier:      {risk['risk_tier']}")
print(f"  Annual premium: ${risk['annual_premium']:,.2f}")
print(f"  Decision:       {risk['decision']}")

## 6. Where the LLMs would go in production

The version above is deliberately minimal. Two natural places to add LLM agents in a real system:

1. **Document agent**: parse uploaded ID, proof of address, and prior policy declarations into   structured fields before enrichment. Reuse the LlamaIndex pattern from Lab 1.
2. **Narrative agent**: produce a one-paragraph rationale for the underwriter to read, citing   the risk score, the enrichment data, and any flags. PydanticAI's structured output is a good   fit here.

Both extensions slot into the same parallel-then-synthesise topology.

## 7. Results

You implemented a parallel-then-synthesise underwriting pipeline in roughly 50 lines of Python. **What to notice about this style:**

* **Parallelism is essentially free** when the work is I/O-bound, which underwriting always is.   Three API calls in 1.2s, not 3s.
* **Failure isolation is one flag away.** `asyncio.gather(..., return_exceptions=True)` lets   you partial-succeed an underwriting run and decide what to do with the missing piece.
* **Sequential + parallel composes cleanly.** This lab's *outer shape* is identical to Lab 1's   claims pipeline — parallel enrichment is just one of the agents in a larger sequence.
* **You can productionise this lab.** Swap the stubs for real APIs, add a narrative agent on   top, and you have a working onboarding workflow.